In [1]:
import graphviz

# Erstellt ein neues gerichtetes Diagramm (Digraph)
# rankdir='LR' sorgt für eine Darstellung von links nach rechts
dot = graphviz.Digraph('EN45555_Recyclability_Calculation', comment='Entity Relationship Diagram')
dot.attr(rankdir='LR', splines='ortho', label='ERD für EN 45555 Recyclability Assessment', fontsize='20')

# Globale Attribute für die Knoten (Entitäten) und Kanten (Beziehungen)
dot.attr('node', shape='record', style='rounded', fontsize='10')
dot.attr('edge', fontsize='8')

# =================================================================================
# Definition der Entitäten als Subgraphen zur besseren Strukturierung
# =================================================================================

# Subgraph 1: Produkt-inhärente Daten (Die erweiterte BOM)
with dot.subgraph(name='cluster_bom') as bom_cluster:
    bom_cluster.attr(style='filled', color='lightgrey', label='BOM Data Inputs (Product-Inherent)')
    
    # Definition der Entität 'Product'
    bom_cluster.node('Product', r'{<head>Product | \l'
                              r'product_id: INTEGER [PK]\l'
                              r'product_name: TEXT\l'
                              r'total_mass_g: REAL\l'
                              r'isArticle: BOOLEAN [cite: 26]\l}',
                     tooltip='Das zu bewertende Gesamtprodukt ')

    # Definition der Entität 'Component' (entspricht ProductPart in IEC 62474)
    bom_cluster.node('Component', r'{<head>Component (ProductPart) | \l'
                                r'component_id: INTEGER [PK]\l'
                                r'component_name: TEXT\l'
                                r'part_mass_g: REAL\l'
                                r'number_of_instances: INTEGER \l'
                                r'is_for_disassembly: BOOLEAN\l}',
                     tooltip='Ein Bauteil oder eine Baugruppe innerhalb des Produkts ')
    
    # Definition der Entität 'JoiningMechanism' - Die empfohlene Erweiterung [cite: 9, 164]
    bom_cluster.node('JoiningMechanism', r'{<head>JoiningMechanism (Custom Extension) | \l'
                                       r'joining_id: INTEGER [PK]\l'
                                       r'joining_type: TEXT [FK]\l'
                                       r'tool_required: TEXT\l'
                                       r'is_reversible: BOOLEAN\l'
                                       r'disassembly_time_s: INTEGER\l}',
                     tooltip='Benutzerdefinierte Erweiterung zur Erfassung von Verbindungs- und Demontage-Daten [cite: 164, 165, 166]')

    # Definition der Entität 'Material'
    bom_cluster.node('Material', r'{<head>Material | \l'
                               r'material_id: INTEGER [PK]\l'
                               r'material_name: TEXT\l'
                               r'material_class: TEXT \l'
                               r'material_mass_g: REAL\l}',
                     tooltip='Ein homogenes Material innerhalb eines Bauteils ')

    # Definition der Entität 'Substance'
    bom_cluster.node('Substance', r'{<head>Substance | \l'
                                r'substance_id: INTEGER [PK]\l'
                                r'substance_name: TEXT\l'
                                r'cas_number: TEXT\l'
                                r'concentration_percent: REAL\l}',
                      tooltip='Eine chemische Substanz innerhalb eines Materials ')

# Subgraph 2: Prozess-abhängige Parameter
with dot.subgraph(name='cluster_eol') as eol_cluster:
    eol_cluster.attr(style='filled', color='lightblue', label='Process-Dependent Parameters')
    
    # Definition der Entität 'EoL_Scenario'
    eol_cluster.node('EoL_Scenario', r'{<head>EoL Scenario | \l'
                                     r'scenario_id: INTEGER [PK]\l'
                                     r'scenario_name: TEXT\l'
                                     r'disassembly_factor_C_dis: REAL\l'
                                     r'sorting_factor_C_sort: REAL\l'
                                     r'recycling_factor_C_recyc: REAL\l}',
                     tooltip='Parameter, die den End-of-Life Prozess beschreiben (z.B. Effizienz der Recyclinganlage)')

# Subgraph 3: Berechnungsergebnis
with dot.subgraph(name='cluster_output') as output_cluster:
    output_cluster.attr(style='filled', color='lightgreen', label='Calculation Output')
    
    # Definition der Entität 'RecyclabilityCalculation'
    output_cluster.node('RecyclabilityCalculation', r'{<head>Recyclability Calculation | \l'
                                                    r'calculation_id: INTEGER [PK]\l'
                                                    r'recyclability_rate_R_cyc: REAL\l'
                                                    r'calculation_date: DATE\l}',
                        tooltip='Das Endergebnis der Berechnung nach EN 45555')


# =================================================================================
# Definition der Beziehungen (Kanten) zwischen den Entitäten
# =================================================================================

# Beziehungen innerhalb der BOM
dot.edge('Product:head', 'Component:head', label='1..n\n(contains)', arrowhead='crow', arrowtail='none', dir='forward')
dot.edge('Component:head', 'JoiningMechanism:head', label='1..n\n(is joined by)', arrowhead='crow', arrowtail='none', dir='forward', style='dashed')
dot.edge('Component:head', 'Material:head', label='1..n\n(is made of)', arrowhead='crow', arrowtail='none', dir='forward')
dot.edge('Material:head', 'Substance:head', label='1..n\n(contains)', arrowhead='crow', arrowtail='none', dir='forward')

# Beziehungen der Inputs zur Berechnung
dot.edge('Product:head', 'RecyclabilityCalculation:head', label='is assessed in', style='dotted')
dot.edge('EoL_Scenario:head', 'RecyclabilityCalculation:head', label='provides parameters for', style='dotted')


# =================================================================================
# Rendern und Speichern des Diagramms
# =================================================================================

# Das Diagramm wird als 'EN45555_ERD.gv' (Source-Code) und 'EN45555_ERD.gv.png' (Bild) gespeichert
output_filename = 'EN45555_ERD.gv'
dot.render(output_filename, view=True, format='png')

print(f"Diagramm wurde erfolgreich als '{output_filename}.png' erstellt und geöffnet.")

Diagramm wurde erfolgreich als 'EN45555_ERD.gv.png' erstellt und geöffnet.
